<a href="https://colab.research.google.com/github/kimdonggyu2008/Personal_Study/blob/main/Encodec.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
!pip install datasets

In [2]:
from datasets import load_dataset, Audio
from transformers import EncodecModel, AutoProcessor

# dummy dataset, however you can swap this with an dataset on the 🤗 hub or bring your own
librispeech_dummy = load_dataset("/content/drive/MyDrive/코딩공부/project_folder/project_dataset/LibriSpeech/train-clean-100", "clean", split="validation")

# load the model + processor (for pre-processing the audio)
model = EncodecModel.from_pretrained("facebook/encodec_24khz")
processor = AutoProcessor.from_pretrained("facebook/encodec_24khz")

# cast the audio data to the correct sampling rate for the model
librispeech_dummy = librispeech_dummy.cast_column("audio", Audio(sampling_rate=processor.sampling_rate))
audio_sample = librispeech_dummy[0]["audio"]["array"]

# pre-process the inputs
inputs = processor(raw_audio=audio_sample, sampling_rate=processor.sampling_rate, return_tensors="pt")

# explicitly encode then decode the audio inputs
encoder_outputs = model.encode(inputs["input_values"], inputs["padding_mask"])
audio_values = model.decode(encoder_outputs.audio_codes, encoder_outputs.audio_scales, inputs["padding_mask"])[0]

# or the equivalent with a forward pass
audio_values = model(inputs["input_values"], inputs["padding_mask"]).audio_values

# you can also extract the discrete codebook representation for LM tasks
# output: concatenated tensor of all the representations
audio_codes = model(inputs["input_values"], inputs["padding_mask"]).audio_codes

Resolving data files:   0%|          | 0/54529 [00:00<?, ?it/s]

ValueError: BuilderConfig 'clean' not found. Available: ['default']

In [3]:
!pip install -U encodec  # stable release
!pip install -U git+https://git@github.com/facebookresearch/encodec#egg=encodec  # bleeding edge
# of if you cloned the repo locally
!pip install .

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.7/3.7 MB 51.9 MB/s eta 0:00:00
  Preparing metadata (setup.py) ... done
  Created wheel for encodec: filename=encodec-0.1.1-py3-none-any.whl size=45759 sha256=520ec00b85a6c8b6ec579e2d96ac58c18de77254694a4e94561f51d8947e3bf0
  Stored in directory: /root/.cache/pip/wheels/b8/eb/9f/e13610cc46ab39d3199fbabebd1c3e142d44b679526e0f228a
Successfully built encodec
  Cloning https://****@github.com/facebookresearch/encodec to /tmp/pip-install-7622bozn/encodec_eb39b0afe378437499d5453588cde6c4
  Running command git clone --filter=blob:none --quiet 'https://****@github.com/facebookresearch/encodec' /tmp/pip-install-7622bozn/encodec_eb39b0afe378437499d5453588cde6c4
  Resolved https://****@github.com/facebookresearch/encodec to commit 0e2d0aed29362c8e8f52494baf3e6f99056b214f
  Preparing metadata (setup.py) ... done
  Created wheel for encodec: filename=encodec-0.1.2a3-py3-none-any.whl size=44544 sha256=39c218a482edb8d6d1dd9e8243992516d56884a3563eb35625e

In [5]:
from encodec import EncodecModel
from encodec.utils import convert_audio

import torchaudio
import torch

# Instantiate a pretrained EnCodec model
model = EncodecModel.encodec_model_24khz()
# The number of codebooks used will be determined bythe bandwidth selected.
# E.g. for a bandwidth of 6kbps, `n_q = 8` codebooks are used.
# Supported bandwidths are 1.5kbps (n_q = 2), 3 kbps (n_q = 4), 6 kbps (n_q = 8) and 12 kbps (n_q =16) and 24kbps (n_q=32).
# For the 48 kHz model, only 3, 6, 12, and 24 kbps are supported. The number
# of codebooks for each is half that of the 24 kHz model as the frame rate is twice as much.
model.set_target_bandwidth(6.0)

# Load and pre-process the audio waveform
wav, sr = torchaudio.load("/content/drive/MyDrive/코딩공부/project_folder/project_dataset/LibriSpeech/test-clean/7176/88083/7176-88083-0027.flac")
wav = convert_audio(wav, sr, model.sample_rate, model.channels)
wav = wav.unsqueeze(0)

# Extract discrete codes from EnCodec
with torch.no_grad():
    encoded_frames = model.encode(wav)
codes = torch.cat([encoded[0] for encoded in encoded_frames], dim=-1)  # [B, n_q, T]

In [6]:
codes

tensor([[[ 475,  779,  887,  ...,  537,  276,  887],
         [ 580,  685,  980,  ...,  685,  870,  870],
         [ 730,  843,  863,  ...,  304, 1000,  480],
         ...,
         [ 885,  814,  683,  ...,   16,  735,  735],
         [ 861,  908,  550,  ...,  767,  550,  712],
         [ 469,  468,  961,  ...,  701,  688,  346]]])

In [8]:
import torch
import torchaudio
from encodec import EncodecModel
from encodec.utils import convert_audio

# -----------------------------------------------------------------------------
# 1. 모델 및 오디오 파일 로드
# -----------------------------------------------------------------------------

# 사전 학습된 EnCodec 24kHz 모델 불러오기
model = EncodecModel.encodec_model_24khz()

# 음질을 결정하는 목표 비트레이트 설정 (e.g., 6.0kbps)
# 이 값에 따라 사용되는 코드북(n_q)의 개수가 결정됩니다. (6.0kbps -> 8개)
model.set_target_bandwidth(6.0)

# 복원할 오디오 파일 경로 (이 부분을 실제 파일 경로로 수정하세요!)
input_audio_path = "/content/drive/MyDrive/코딩공부/project_folder/project_dataset/LibriSpeech/test-clean/7176/88083/7176-88083-0027.flac"
output_audio_path = "/content/drive/MyDrive/restored_audio.wav"

# torchaudio로 오디오 파일 로드
original_wav, sr = torchaudio.load(input_audio_path)

# -----------------------------------------------------------------------------
# 2. 전처리 (Preprocessing)
# -----------------------------------------------------------------------------

# 오디오를 모델의 입력 형식에 맞게 변환
# (샘플링 레이트, 채널 수를 모델에 맞춤)
wav_for_encoding = convert_audio(original_wav, sr, model.sample_rate, model.channels)

# 모델은 배치(batch) 입력을 가정하므로, 배치 차원 추가 [C, T] -> [B, C, T]
# 여기서 B=1
wav_for_encoding = wav_for_encoding.unsqueeze(0)

print(f"오디오 로드 및 전처리 완료. 입력 텐서 형태: {wav_for_encoding.shape}")

# -----------------------------------------------------------------------------
# 3. 인코딩 (Audio -> Codes)
# -----------------------------------------------------------------------------

# torch.no_grad()를 사용해 불필요한 그래디언트 계산 방지
with torch.no_grad():
    print("인코딩을 시작합니다...")
    # encoded_frames는 [(codes, scale), ...] 형태의 리스트입니다.
    # 이 변수는 디코딩에 그대로 사용되므로 가공하면 안 됩니다.
    encoded_frames = model.encode(wav_for_encoding)
    print("인코딩 완료!")

# 참고: 만약 '토큰'만 따로 보고 싶다면 아래 코드를 사용
codes_only = torch.cat([encoded[0] for encoded in encoded_frames], dim=-1)
print(f"추출된 오디오 토큰의 형태: {codes_only.shape}")
print(codes_only)

# -----------------------------------------------------------------------------
# 4. 디코딩 (Codes -> Audio)
# -----------------------------------------------------------------------------

with torch.no_grad():
    print("\n디코딩을 시작합니다...")
    # model.encode()가 반환한 encoded_frames를 그대로 decode 함수에 전달
    restored_wav = model.decode(encoded_frames)
    print("디코딩 완료!")


# -----------------------------------------------------------------------------
# 5. 파일로 저장 및 확인
# -----------------------------------------------------------------------------

# 복원된 오디오 파형을 .wav 파일로 저장
# 배치 차원을 제거하고 CPU로 이동
torchaudio.save(output_audio_path, restored_wav.squeeze(0).cpu(), model.sample_rate)

print(f"\n'{output_audio_path}' 파일로 복원이 완료되었습니다.")
print("원본 오디오와 복원된 오디오를 비교해서 들어보세요!")

오디오 로드 및 전처리 완료. 입력 텐서 형태: torch.Size([1, 1, 51120])
인코딩을 시작합니다...
인코딩 완료!
추출된 오디오 토큰의 형태: torch.Size([1, 8, 160])
tensor([[[ 475,  779,  887,  ...,  537,  276,  887],
         [ 580,  685,  980,  ...,  685,  870,  870],
         [ 730,  843,  863,  ...,  304, 1000,  480],
         ...,
         [ 885,  814,  683,  ...,   16,  735,  735],
         [ 861,  908,  550,  ...,  767,  550,  712],
         [ 469,  468,  961,  ...,  701,  688,  346]]])

디코딩을 시작합니다...
디코딩 완료!

'/content/drive/MyDrive/restored_audio.wav' 파일로 복원이 완료되었습니다.
원본 오디오와 복원된 오디오를 비교해서 들어보세요!
